## **First Set up environment to run hybrid (CUDA+MPI) model in Colab:**

In [1]:
!nvidia-smi   # verify gpu availability

Mon Jan  6 10:35:25 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!apt-get update                                   # Update and install dependencies
!apt-get install -y libopenmpi-dev openmpi-bin    # Installing MPI libraries
!apt-get install -y nvidia-cuda-toolkit           # Installing the CUDA Toolkit
!pip install mpi4py                               # Installing MPI for Python for Python-MPI communication


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,197 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,632 kB]
Fetched 4,225 kB in 4s (1,063 kB/s)
Reading pack

# **Complex Engineering Porblem:**

---
Hybrid parallel computing frameworks that integrates:
*   MPI for inter-node communication.
*   CUDA for GPU-accelerated intra-node computation.








## **Hybrid (CUDA+MPI) Model for Brightness Adjustment and Contrast Enhancement:**

In [3]:
%%writefile mpi_cuda_cep1.cu

#include <mpi.h>
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define WIDTH 5    // Image width
#define HEIGHT 5   // Image height
#define BLOCK_SIZE 2  // CUDA block size

// CUDA kernel for brightness adjustment
__global__ void adjust_brightness(int* image, int width, int height, int brightness_value) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int idy = blockIdx.y * blockDim.y + threadIdx.y;

    if (idx < width && idy < height) {
        int index = idy * width + idx;
        image[index] += brightness_value;
        // Clamp values between 0 and 255
        image[index] = min(max(image[index], 0), 255);
    }
}

// CUDA kernel for contrast enhancement
__global__ void enhance_contrast(int* image, int width, int height, float contrast_factor) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int idy = blockIdx.y * blockDim.y + threadIdx.y;

    if (idx < width && idy < height) {
        int index = idy * width + idx;
        float adjusted = (image[index] - 128) * contrast_factor + 128;
        // Clamp values between 0 and 255
        image[index] = min(max((int)adjusted, 0), 255);
    }
}

// Initialize an image with random values
void init_image(int* image, int width, int height) {
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            image[i * width + j] = rand() % 256; // Values between 0 and 255
        }
    }
}

// Print an image
void print_image(int* image, int width, int height) {
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            printf("%3d ", image[i * width + j]);
        }
        printf("\n");
    }
}

int main(int argc, char** argv) {
    int rank, size;

    // Initialize MPI
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int* image = NULL;
    int* sub_image = NULL;
    int* d_sub_image = NULL;

    int brightness_value = 10; // Brightness adjustment value
    float contrast_factor = 1.5;  // Contrast enhancement factor
    int chunk_size = (WIDTH * HEIGHT) / size;

    if (rank == size - 1) {
        chunk_size += (WIDTH * HEIGHT) % size;  // Handle remainder for last process
    }

    // Allocate memory for sub_image
    sub_image = (int*)malloc(chunk_size * sizeof(int));

    // Initialize image data on the master node (rank 0)
    if (rank == 0)
    {
        image = (int*)malloc(WIDTH * HEIGHT * sizeof(int));
        init_image(image, WIDTH, HEIGHT);

        printf("\n<<<<----- Task: Brightness Adjustment and Contrast Enhancement on Original Image (Rank: %d) ---------->>>>\n\n", rank);
        printf("Original Image:\n");
        print_image(image, WIDTH, HEIGHT);

        // Add Header Info for Brightness Adjustment Task
        printf("\n<<<<-------------------------------------------------------------------------------->>>>\n");
        printf("<<<<------------------------ Brightness Adjustment -------------------------------->>>>\n");
        printf("<<<<------------------------------------------------------------------------------->>>>\n\n");
    }

    // Scatter the image to all processes for brightness adjustment
    MPI_Scatter(image, chunk_size, MPI_INT, sub_image, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    // Display received data
    printf("Process %d received data for brightness adjustment: ", rank);
    for (int i = 0; i < chunk_size; i++) {
        printf("%3d ", sub_image[i]);
    }
    printf("\n");

    // Allocate memory on the GPU
    cudaMalloc((void**)&d_sub_image, chunk_size * sizeof(int));

    // Copy data to GPU for brightness adjustment
    cudaMemcpy(d_sub_image, sub_image, chunk_size * sizeof(int), cudaMemcpyHostToDevice);

    // Launch brightness adjustment kernel
    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((WIDTH + block.x - 1) / block.x, (HEIGHT + block.y - 1) / block.y);

    adjust_brightness<<<grid, block>>>(d_sub_image, WIDTH, HEIGHT, brightness_value);

    cudaDeviceSynchronize();
    cudaMemcpy(sub_image, d_sub_image, chunk_size * sizeof(int), cudaMemcpyDeviceToHost);

    // Gather results after brightness adjustment
    MPI_Gather(sub_image, chunk_size, MPI_INT, image, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("\nImage after Brightness Adjustment:\n");
        print_image(image, WIDTH, HEIGHT);
        printf("\n");

        // Add Header Info for Contrast Enhancement Task
        printf("\n<<<<-------------------------------------------------------------------------------->>>>\n");
        printf("<<<<------------------------ Contrast Enhancement ----------------------------------->>>>\n");
        printf("<<<<--------------------------------------------------------------------------------->>>>\n\n");
    }

    // Scatter the original image again for contrast enhancement
    MPI_Scatter(image, chunk_size, MPI_INT, sub_image, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    // Display received data for contrast enhancement
    printf("Process %d received data for contrast enhancement: ", rank);
    for (int i = 0; i < chunk_size; i++) {
        printf("%3d ", sub_image[i]);
    }
    printf("\n");

    // Copy data to GPU for contrast enhancement
    cudaMemcpy(d_sub_image, sub_image, chunk_size * sizeof(int), cudaMemcpyHostToDevice);

    // Launch contrast enhancement kernel
    enhance_contrast<<<grid, block>>>(d_sub_image, WIDTH, HEIGHT, contrast_factor);

    cudaDeviceSynchronize();
    cudaMemcpy(sub_image, d_sub_image, chunk_size * sizeof(int), cudaMemcpyDeviceToHost);

    // Gather contrast-enhanced results
    MPI_Gather(sub_image, chunk_size, MPI_INT, image, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("\nImage after Contrast Enhancement:\n");
        print_image(image, WIDTH, HEIGHT);
        printf("\n");
    }

    // Clean up
    if (rank == 0) {
        free(image);
    }
    free(sub_image);
    cudaFree(d_sub_image);

    MPI_Finalize();
    return 0;
}


Writing mpi_cuda_cep1.cu


In [4]:
!nvcc -o project mpi_cuda_cep1.cu -I/usr/lib/x86_64-linux-gnu/openmpi/include -L/usr/lib/x86_64-linux-gnu -lmpi -lcudart


!mpirun --allow-run-as-root --oversubscribe -np 5 ./project


<<<<----- Task: Brightness Adjustment and Contrast Enhancement on Original Image (Rank: 0) ---------->>>>

Original Image:
103 198 105 115  81 
255  74 236  41 205 
186 171 242 251 227 
 70 124 194  84 248 
 27 232 231 141 118 

<<<<-------------------------------------------------------------------------------->>>>
<<<<------------------------ Brightness Adjustment -------------------------------->>>>
<<<<------------------------------------------------------------------------------->>>>

Process 0 received data for brightness adjustment: 103 198 105 115  81 
Process 2 received data for brightness adjustment: 186 171 242 251 227 
Process 1 received data for brightness adjustment: 255  74 236  41 205 
Process 3 received data for brightness adjustment:  70 124 194  84 248 
Process 4 received data for brightness adjustment:  27 232 231 141 118 

Image after Brightness Adjustment:
113 208 115 125  91 
255  84 246  51 215 
196 181 252 255 237 
 80 134 204  94 255 
 37 242 241 151 128 


<

## **Hybrid (CUDA+MPI) Model for Convolution:**

In [5]:
%%writefile mpi_cuda_2.cu

#include <mpi.h>
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define WIDTH 5    // Image width
#define HEIGHT 5   // Image height
#define BLOCK_SIZE 2  // CUDA block size

// Convolution Kernel (3x3) for edge detection
__constant__ int kernel[3][3] = {
    { -1, -1, -1 },
    { -1,  8, -1 },
    { -1, -1, -1 }
};

// CUDA kernel for convolution
__global__ void apply_convolution(int* image, int* result, int width, int height) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int idy = blockIdx.y * blockDim.y + threadIdx.y;

    if (idx < width && idy < height) {
        int sum = 0;

        // Apply the 3x3 kernel to the image
        for (int ky = -1; ky <= 1; ky++) {
            for (int kx = -1; kx <= 1; kx++) {
                int ix = min(max(idx + kx, 0), width - 1);
                int iy = min(max(idy + ky, 0), height - 1);
                int image_idx = iy * width + ix;
                int kernel_val = kernel[ky + 1][kx + 1];
                sum += image[image_idx] * kernel_val;
            }
        }

        // Clamp result between 0 and 255
        result[idy * width + idx] = min(max(sum, 0), 255);
        //result[idy * width + idx] = sum;
    }
}

// Initialize an image with random values
void init_image(int* image, int width, int height) {
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            image[i * width + j] = rand() % 256; // Values between 0 and 255
        }
    }
}

// Print an image
void print_image(int* image, int width, int height) {
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            printf("%3d ", image[i * width + j]);
        }
        printf("\n");
    }
}

int main(int argc, char** argv) {
    int rank, size;

    // Initialize MPI
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int* image = NULL;
    int* result = NULL;
    int* sub_image = NULL;
    int* d_sub_image = NULL;
    int* d_result = NULL;

    int chunk_size = (WIDTH * HEIGHT) / size;

    if (rank == size - 1) {
        chunk_size += (WIDTH * HEIGHT) % size;  // Handle remainder for last process
    }

    // Allocate memory for sub_image and result
    sub_image = (int*)malloc(chunk_size * sizeof(int));
    result = (int*)malloc(WIDTH * HEIGHT * sizeof(int));

    // Initialize image data on the master node (rank 0)
    if (rank == 0) {
        image = (int*)malloc(WIDTH * HEIGHT * sizeof(int));
        init_image(image, WIDTH, HEIGHT);

        printf("\n<<<<----- Task: Convolution on Original Image (Rank: %d) ---------->>>>\n\n", rank);
        printf("Original Image:\n");
        print_image(image, WIDTH, HEIGHT);

        // Add Header Info for Convolution Task
        printf("\n<<<<-------------------------------------------------------------------------------->>>>\n");
        printf("<<<<------------------------ Convolution ---------------------------------------->>>>\n");
        printf("<<<<------------------------------------------------------------------------------->>>>\n\n");
    }

    // Scatter the image to all processes
    MPI_Scatter(image, chunk_size, MPI_INT, sub_image, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    // Display received data
    printf("Process %d received data for convolution: ", rank);
    for (int i = 0; i < chunk_size; i++) {
        printf("%3d ", sub_image[i]);
    }
    printf("\n");

    // Allocate memory on the GPU
    cudaMalloc((void**)&d_sub_image, chunk_size * sizeof(int));
    cudaMalloc((void**)&d_result, WIDTH * HEIGHT * sizeof(int));

    // Copy data to GPU
    cudaMemcpy(d_sub_image, sub_image, chunk_size * sizeof(int), cudaMemcpyHostToDevice);

    // Launch convolution kernel
    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((WIDTH + block.x - 1) / block.x, (HEIGHT + block.y - 1) / block.y);

    apply_convolution<<<grid, block>>>(d_sub_image, d_result, WIDTH, HEIGHT);

    cudaDeviceSynchronize();

    // Copy the result back to host
    cudaMemcpy(result, d_result, WIDTH * HEIGHT * sizeof(int), cudaMemcpyDeviceToHost);

    // Gather results
    MPI_Gather(result, chunk_size, MPI_INT, image, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("\nImage after Convolution:\n");
        print_image(image, WIDTH, HEIGHT);
        printf("\n");
    }

    // Clean up
    if (rank == 0) {
        free(image);
    }
    free(sub_image);
    free(result);
    cudaFree(d_sub_image);
    cudaFree(d_result);

    MPI_Finalize();
    return 0;
}


Writing mpi_cuda_2.cu


In [6]:
!nvcc -o project mpi_cuda_2.cu -I/usr/lib/x86_64-linux-gnu/openmpi/include -L/usr/lib/x86_64-linux-gnu -lmpi -lcudart
!mpirun --allow-run-as-root --oversubscribe -np 5 ./project


<<<<----- Task: Convolution on Original Image (Rank: 0) ---------->>>>

Original Image:
103 198 105 115  81 
255  74 236  41 205 
186 171 242 251 227 
 70 124 194  84 248 
 27 232 231 141 118 

<<<<-------------------------------------------------------------------------------->>>>
<<<<------------------------ Convolution ---------------------------------------->>>>
<<<<------------------------------------------------------------------------------->>>>

Process 0 received data for convolution: 103 198 105 115  81 
Process 4 received data for convolution:  27 232 231 141 118 
Process 1 received data for convolution: 255  74 236  41 205 
Process 2 received data for convolution: 186 171 242 251 227 
Process 3 received data for convolution:  70 124 194  84 248 

Image after Convolution:
119 255 109 255 175 
255   0 255   0 255 
255 255 255 255 255 
102 255 255   0 255 
  0 255 255 255 255 



## **Hybrid (CUDA+MPI) Model for Matrix Traspose:**

In [7]:
%%writefile mpi_cuda_3.cu

#include <mpi.h>
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define WIDTH 5    // Matrix width
#define HEIGHT 5   // Matrix height
#define BLOCK_SIZE 2  // CUDA block size

// CUDA kernel for matrix transpose
__global__ void transpose(int* matrix, int* transposed_matrix, int width, int height)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < height && col < width) {
        transposed_matrix[col * height + row] = matrix[row * width + col];
    }
}

// Initialize a matrix with random values
void init_matrix(int* matrix, int width, int height) {
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            matrix[i * width + j] = rand() % 256;
        }
    }
}

// Print a matrix
void print_matrix(int* matrix, int width, int height) {
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            printf("%3d ", matrix[i * width + j]);
        }
        printf("\n");
    }
}

int main(int argc, char** argv) {
    int rank, size;

    // Initialize MPI
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int* matrix = NULL;
    int* transposed_matrix = NULL;
    int* sub_matrix = NULL;
    int* transposed_sub_matrix = NULL;
    int* d_matrix = NULL;
    int* d_transposed_matrix = NULL;

    int rows_per_process = HEIGHT / size;
    int chunk_size = rows_per_process * WIDTH;

    if (rank == 0)
    {
        printf("<<<<----- Task: Transpose of Original Image (Rank: %d) ---------->>>>\n\n", rank);
        matrix = (int*)malloc(WIDTH * HEIGHT * sizeof(int));
        transposed_matrix = (int*)malloc(WIDTH * HEIGHT * sizeof(int));
        init_matrix(matrix, WIDTH, HEIGHT);

        // Add Header Info for transpose Task
        printf("\n<<<<-------------------------------------------------------------------------------->>>>\n");
        printf("<<<<------------------------ Matrix Transpose ---------------------------------------->>>>\n");
        printf("<<<<------------------------------------------------------------------------------->>>>\n\n");


        printf("Original Matrix:\n");
        print_matrix(matrix, WIDTH, HEIGHT);
    }
    printf("\n");

    sub_matrix = (int*)malloc(chunk_size * sizeof(int));
    transposed_sub_matrix = (int*)malloc(chunk_size * sizeof(int));

    MPI_Scatter(matrix, chunk_size, MPI_INT, sub_matrix, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    printf("Rank %d received chunk:\n", rank);
    for (int i = 0; i < rows_per_process; i++) {
        for (int j = 0; j < WIDTH; j++) {
            printf("%3d ", sub_matrix[i * WIDTH + j]);
        }
        printf("\n");
    }
    printf("\n");

    cudaMalloc((void**)&d_matrix, chunk_size * sizeof(int));
    cudaMalloc((void**)&d_transposed_matrix, chunk_size * sizeof(int));

    cudaMemcpy(d_matrix, sub_matrix, chunk_size * sizeof(int), cudaMemcpyHostToDevice);

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((WIDTH + block.x - 1) / block.x, (rows_per_process + block.y - 1) / block.y);

    transpose<<<grid, block>>>(d_matrix, d_transposed_matrix, WIDTH, rows_per_process);
    cudaDeviceSynchronize();

    cudaMemcpy(transposed_sub_matrix, d_transposed_matrix, chunk_size * sizeof(int), cudaMemcpyDeviceToHost);

    // Gather the transposed data
    MPI_Gather(transposed_sub_matrix, chunk_size, MPI_INT, transposed_matrix, chunk_size, MPI_INT, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("Gathered transposed data (before reorganization):\n");
        for (int i = 0; i < HEIGHT; i++) {
            for (int j = 0; j < WIDTH; j++) {
                printf("%3d ", transposed_matrix[i * WIDTH + j]);
            }
            printf("\n");
        }

        // Reorganize gathered data into the correct transposed format
        int* final_transposed_matrix = (int*)malloc(WIDTH * HEIGHT * sizeof(int));
        for (int i = 0; i < HEIGHT; i++) {
            for (int j = 0; j < WIDTH; j++) {
                final_transposed_matrix[j * HEIGHT + i] = transposed_matrix[i * WIDTH + j];
            }
        }

        printf("\nFinal Transposed Matrix:\n");
        print_matrix(final_transposed_matrix, HEIGHT, WIDTH);

        free(final_transposed_matrix);
    }

    // Clean up
    if (rank == 0) {
        free(matrix);
        free(transposed_matrix);
    }
    free(sub_matrix);
    free(transposed_sub_matrix);
    cudaFree(d_matrix);
    cudaFree(d_transposed_matrix);

    MPI_Finalize();
    return 0;
}


Writing mpi_cuda_3.cu


In [8]:
!nvcc -o task mpi_cuda_3.cu -I/usr/lib/x86_64-linux-gnu/openmpi/include -L/usr/lib/x86_64-linux-gnu -lmpi -lcudart
!mpirun --allow-run-as-root --oversubscribe -np 5 ./task

<<<<----- Task: Transpose of Original Image (Rank: 0) ---------->>>>


<<<<-------------------------------------------------------------------------------->>>>
<<<<------------------------ Matrix Transpose ---------------------------------------->>>>
<<<<------------------------------------------------------------------------------->>>>

Original Matrix:
103 198 105 115  81 
255  74 236  41 205 
186 171 242 251 227 
 70 124 194  84 248 
 27 232 231 141 118 

Rank 0 received chunk:
103 198 105 115  81 


Rank 1 received chunk:
255  74 236  41 205 


Rank 2 received chunk:
186 171 242 251 227 


Rank 3 received chunk:
 70 124 194  84 248 


Rank 4 received chunk:
 27 232 231 141 118 

Gathered transposed data (before reorganization):
103 198 105 115  81 
255  74 236  41 205 
186 171 242 251 227 
 70 124 194  84 248 
 27 232 231 141 118 

Final Transposed Matrix:
103 255 186  70  27 
198  74 171 124 232 
105 236 242 194 231 
115  41 251  84 141 
 81 205 227 248 118 


## **Hybrid (CUDA+MPI) Model for Matrix Multiplication:**

In [9]:
%%writefile mpi_cuda_4.cu

#include <mpi.h>
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define WIDTH 5       // Matrix width (columns)
#define HEIGHT 5      // Matrix height (rows)
#define BLOCK_SIZE 2  // CUDA block size

// CUDA kernel for matrix multiplication
__global__ void matrixMultiply(int* A_row, int* B, int* C, int width, int height)
{
    int row = blockIdx.x;   // Row index in C
    int col = threadIdx.x;  // Column index in C

    if (row < height && col < width) {
        int sum = 0;
        for (int k = 0; k < width; k++) {
            sum += A_row[row * width + k] * B[k * width + col];
        }
        C[row * width + col] = sum;
    }
}

// Initialize a matrix with random values
void init_matrix(int* matrix, int width, int height)
{
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            matrix[i * width + j] = rand() % 10;
        }
    }
}

// Print a matrix
void print_matrix(int* matrix, int width, int height)
{
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            printf("%d ", matrix[i * width + j]);
        }
        printf("\n");
    }
}

int main(int argc, char** argv) {
    int rank, size;

    // Initialize MPI
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int* A = NULL;
    int* B = NULL;
    int* C = NULL;
    int* A_local = NULL;
    int* C_local = NULL;

    int rows_per_process = HEIGHT / size;
    int extra_rows = HEIGHT % size;
    int local_rows = (rank < extra_rows) ? rows_per_process + 1 : rows_per_process;

    if (rank == 0)
    {
        printf("\n<<<<------------- Task: Matrix Multiplications (Rank: %d) ------------------>>>>\n\n", rank);

        A = (int*)malloc(WIDTH * HEIGHT * sizeof(int));
        B = (int*)malloc(WIDTH * WIDTH * sizeof(int));
        C = (int*)malloc(WIDTH * HEIGHT * sizeof(int));

        init_matrix(A, WIDTH, HEIGHT);
        init_matrix(B, WIDTH, WIDTH);

        // Add Header Info for Matrix Multiplication Task
        printf("\n<<<<-------------------------------------------------------------------------------->>>>\n");
        printf("<<<<------------------------ Matrix Multiplications ---------------------------------->>>>\n");
        printf("<<<<------------------------------------------------------------------------------->>>>\n\n");

        printf("Matrix A:\n");
        print_matrix(A, WIDTH, HEIGHT);

        printf("\nMatrix B:\n");
        print_matrix(B, WIDTH, WIDTH);
    }

    // Broadcast B matrix to all processes
    if (rank == 0)
    {
        MPI_Bcast(B, WIDTH * WIDTH, MPI_INT, 0, MPI_COMM_WORLD);
    }
    else {
        B = (int*)malloc(WIDTH * WIDTH * sizeof(int));
        MPI_Bcast(B, WIDTH * WIDTH, MPI_INT, 0, MPI_COMM_WORLD);
    }

    // Allocate memory for local submatrix
    A_local = (int*)malloc(local_rows * WIDTH * sizeof(int));
    C_local = (int*)malloc(local_rows * WIDTH * sizeof(int));

    // Distribute rows of A to all processes
    int* sendcounts = (int*)malloc(size * sizeof(int));
    int* displs = (int*)malloc(size * sizeof(int));

    int offset = 0;
    for (int i = 0; i < size; i++) {
        sendcounts[i] = (i < extra_rows) ? (rows_per_process + 1) * WIDTH : rows_per_process * WIDTH;
        displs[i] = offset;
        offset += sendcounts[i];
    }

    MPI_Scatterv(A, sendcounts, displs, MPI_INT, A_local, local_rows * WIDTH, MPI_INT, 0, MPI_COMM_WORLD);

    printf("\nRank %d received %d rows: ", rank, local_rows);
    for (int i = 0; i < local_rows; i++) {
        for (int j = 0; j < WIDTH; j++) {
            printf("%d ", A_local[i * WIDTH + j]);
        }
        printf("\n");
    }

    // Allocate device memory for CUDA
    int* d_A_local;
    int* d_B;
    int* d_C_local;
    cudaMalloc((void**)&d_A_local, local_rows * WIDTH * sizeof(int));
    cudaMalloc((void**)&d_B, WIDTH * WIDTH * sizeof(int));
    cudaMalloc((void**)&d_C_local, local_rows * WIDTH * sizeof(int));

    // Copy data from host to device
    cudaMemcpy(d_A_local, A_local, local_rows * WIDTH * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, WIDTH * WIDTH * sizeof(int), cudaMemcpyHostToDevice);

    // Define block and grid sizes
    dim3 block(WIDTH, 1);
    dim3 grid(local_rows, 1);

    // Launch CUDA kernel for matrix multiplication
    matrixMultiply<<<grid, block>>>(d_A_local, d_B, d_C_local, WIDTH, local_rows);

    // Copy result back to host
    cudaMemcpy(C_local, d_C_local, local_rows * WIDTH * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nRank %d computed local results: ", rank);
    for (int i = 0; i < local_rows; i++) {
        for (int j = 0; j < WIDTH; j++) {
            printf("%d ", C_local[i * WIDTH + j]);
        }
    }
    printf("\n");

    // Gather results from all processes
    MPI_Gatherv(C_local, local_rows * WIDTH, MPI_INT, C, sendcounts, displs, MPI_INT, 0, MPI_COMM_WORLD);

    // Rank 0 prints the result matrix
    if (rank == 0)
    {
        printf("\nMatrix C after multiplication (C = A * B):\n");
        print_matrix(C, WIDTH, HEIGHT);

        free(A);
        free(B);
        free(C);
    }

    free(A_local);
    free(C_local);
    cudaFree(d_A_local);
    cudaFree(d_B);
    cudaFree(d_C_local);
    free(sendcounts);
    free(displs);

    MPI_Finalize();
    return 0;
}


Writing mpi_cuda_4.cu


In [10]:
!nvcc -o project mpi_cuda_4.cu -I/usr/lib/x86_64-linux-gnu/openmpi/include -L/usr/lib/x86_64-linux-gnu -lmpi -lcudart
!mpirun --allow-run-as-root --oversubscribe -np 5 ./project


<<<<------------- Task: Matrix Multiplications (Rank: 0) ------------------>>>>


<<<<-------------------------------------------------------------------------------->>>>
<<<<------------------------ Matrix Multiplications ---------------------------------->>>>
<<<<------------------------------------------------------------------------------->>>>

Matrix A:
3 6 7 5 3 
5 6 2 9 1 
2 7 0 9 3 
6 0 6 2 6 
1 8 7 9 2 

Matrix B:
0 2 3 7 5 
9 2 2 8 9 
7 3 6 1 2 
9 3 1 9 4 
7 8 4 5 0 

Rank 0 received 1 rows: 3 6 7 5 3 

Rank 1 received 1 rows: 5 6 2 9 1 

Rank 3 received 1 rows: 6 0 6 2 6 

Rank 2 received 1 rows: 2 7 0 9 3 

Rank 4 received 1 rows: 1 8 7 9 2 

Rank 0 computed local results: 169 78 80 136 103 

Rank 1 computed local results: 156 63 52 171 119 

Rank 3 computed local results: 102 84 80 96 50 

Rank 4 computed local results: 216 82 78 169 127 

Matrix C after multiplication (C = A * B):
169 78 80 136 103 
156 63 52 171 119 
165 69 41 166 109 
102 84 80 96 50 
216 82 78 169 127

**Performance Evaluation of Matrix Multiplication of Hybrid (MPI + CUDA) and Traditional (single-CPU):**

In [11]:
%%writefile per1.cu

#include <mpi.h>
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH 1024       // Matrix width
#define HEIGHT 1024      // Matrix height
#define BLOCK_SIZE 16  // CUDA block size

// CUDA kernel for matrix multiplication
__global__ void matrixMultiply(int* A_row, int* B, int* C, int width, int height)
{
    int row = blockIdx.x;   // Row index in C
    int col = threadIdx.x;  // Column index in C

    if (row < height && col < width) {
        int sum = 0;
        for (int k = 0; k < width; k++) {
            sum += A_row[row * width + k] * B[k * width + col];
        }
        C[row * width + col] = sum;
    }
}

// Traditional (Single CPU) matrix multiplication
void matrixMultiplyTraditional(int* A, int* B, int* C, int width, int height)
{
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            int sum = 0;
            for (int k = 0; k < width; k++) {
                sum += A[i * width + k] * B[k * width + j];
            }
            C[i * width + j] = sum;
        }
    }
}

// Initialize a matrix
void init_matrix(int* matrix, int width, int height)
{
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            matrix[i * width + j] = rand() % 10;
        }
    }
}

// Print a matrix
void print_matrix(int* matrix, int width, int height)
{
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            printf("%d ", matrix[i * width + j]);
        }
        printf("\n");
    }
}

int main(int argc, char** argv)
{
    int rank, size;

    // Initialize MPI
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int* A = NULL;
    int* B = NULL;
    int* C = NULL;
    int* A_local = NULL;
    int* C_local = NULL;

    int rows_per_process = HEIGHT / size;
    int extra_rows = HEIGHT % size;
    int local_rows = (rank < extra_rows) ? rows_per_process + 1 : rows_per_process;

    if (rank == 0)
    {
        printf("\n<<<<------------- Task: Matrix Multiplications Performance Evaluations (Rank: %d) ------------------>>>>\n\n", rank);

        A = (int*)malloc(WIDTH * HEIGHT * sizeof(int));
        B = (int*)malloc(WIDTH * WIDTH * sizeof(int));
        C = (int*)malloc(WIDTH * HEIGHT * sizeof(int));

        init_matrix(A, WIDTH, HEIGHT);
        init_matrix(B, WIDTH, WIDTH);

        // Print matrices
        //printf("Matrix A:\n");
        //print_matrix(A, WIDTH, HEIGHT);
        //printf("\nMatrix B:\n");
        //print_matrix(B, WIDTH, WIDTH);
    }

    // Broadcast B matrix to all processes
    if (rank == 0)
    {
        MPI_Bcast(B, WIDTH * WIDTH, MPI_INT, 0, MPI_COMM_WORLD);
    }
    else {
        B = (int*)malloc(WIDTH * WIDTH * sizeof(int));
        MPI_Bcast(B, WIDTH * WIDTH, MPI_INT, 0, MPI_COMM_WORLD);
    }

    // Allocate memory for local submatrix
    A_local = (int*)malloc(local_rows * WIDTH * sizeof(int));
    C_local = (int*)malloc(local_rows * WIDTH * sizeof(int));

    // Distribute rows of A to all processes
    int* sendcounts = (int*)malloc(size * sizeof(int));
    int* displs = (int*)malloc(size * sizeof(int));

    int offset = 0;
    for (int i = 0; i < size; i++) {
        sendcounts[i] = (i < extra_rows) ? (rows_per_process + 1) * WIDTH : rows_per_process * WIDTH;
        displs[i] = offset;
        offset += sendcounts[i];
    }

    MPI_Scatterv(A, sendcounts, displs, MPI_INT, A_local, local_rows * WIDTH, MPI_INT, 0, MPI_COMM_WORLD);

    // Allocate device memory for CUDA
    int* d_A_local;
    int* d_B;
    int* d_C_local;
    cudaMalloc((void**)&d_A_local, local_rows * WIDTH * sizeof(int));
    cudaMalloc((void**)&d_B, WIDTH * WIDTH * sizeof(int));
    cudaMalloc((void**)&d_C_local, local_rows * WIDTH * sizeof(int));

    // Copy data from host to device
    cudaMemcpy(d_A_local, A_local, local_rows * WIDTH * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, WIDTH * WIDTH * sizeof(int), cudaMemcpyHostToDevice);

    // Define block and grid sizes
    dim3 block(WIDTH, 1);
    dim3 grid(local_rows, 1);

    // Measure the execution time of hybrid (CUDA+MPI)
    double start_time = MPI_Wtime();

    // Launch CUDA kernel for matrix multiplication
    matrixMultiply<<<grid, block>>>(d_A_local, d_B, d_C_local, WIDTH, local_rows);

    // Copy result back to host
    cudaMemcpy(C_local, d_C_local, local_rows * WIDTH * sizeof(int), cudaMemcpyDeviceToHost);

    // Gather results from all processes
    MPI_Gatherv(C_local, local_rows * WIDTH, MPI_INT, C, sendcounts, displs, MPI_INT, 0, MPI_COMM_WORLD);

    double end_time = MPI_Wtime();
    double hybrid_time = end_time - start_time;

    // Rank 0 prints the result matrix for hybrid
    if (rank == 0) {
        //printf("\nMatrix C after hybrid multiplication (C = A * B):\n");
        //print_matrix(C, WIDTH, HEIGHT);
    }

    // Traditional (CPU) matrix multiplication measurement
    double traditional_start_time = MPI_Wtime();

    if (rank == 0) {
        matrixMultiplyTraditional(A, B, C, WIDTH, HEIGHT);
    }

    double traditional_end_time = MPI_Wtime();
    double traditional_time = traditional_end_time - traditional_start_time;

    // Rank 0 prints the result matrix for traditional
    if (rank == 0) {
        //printf("\nMatrix C after traditional multiplication (C = A * B):\n");
        //print_matrix(C, WIDTH, HEIGHT);

        // Speedup calculation
        double speedup = traditional_time / hybrid_time;

        printf("\nExecution Time (Hybrid (Cuda+MPI)): %f seconds\n", hybrid_time);
        printf("Execution Time (Traditional (Single-CPU)): %f seconds\n", traditional_time);
        printf("Speedup (Traditional / Hybrid): %f\n", speedup);
    }

    // Cleanup
    free(A_local);
    free(C_local);
    cudaFree(d_A_local);
    cudaFree(d_B);
    cudaFree(d_C_local);
    free(sendcounts);
    free(displs);

    if (rank == 0) {
        free(A);
        free(B);
        free(C);
    }

    MPI_Finalize();
    return 0;
}


Writing per1.cu


In [12]:
!nvcc -o project per1.cu -I/usr/lib/x86_64-linux-gnu/openmpi/include -L/usr/lib/x86_64-linux-gnu -lmpi -lcudart
!mpirun --allow-run-as-root --oversubscribe -np 5 ./project


<<<<------------- Task: Matrix Multiplications Performance Evaluations (Rank: 0) ------------------>>>>


Execution Time (Hybrid (Cuda+MPI)): 0.013263 seconds
Execution Time (Traditional (Single-CPU)): 10.922911 seconds
Speedup (Traditional / Hybrid): 823.571447
